# 沪深300数据下载与本地化

优先读取冻结 CSV 与 Parquet；文件不存在时才请求网络。已有数据不覆盖。冻结成分股日期为2026-09-04，不代表历史成分股。供应商后续修订可能使重新下载的行情不同。

In [1]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/data.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook from the project root or notebooks directory")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [2]:
import akshare as ak
import pandas as pd
from pathlib import Path

In [3]:
data_dir = ROOT / "data/raw"
data_dir.mkdir(parents=True, exist_ok=True)

In [4]:
cons_path = data_dir / "csi300_constituents.csv"
cons = pd.read_csv(cons_path, dtype=str) if cons_path.exists() else ak.index_stock_cons_csindex(symbol="000300")
cons.head()

Out[0]: 
           日期    指数代码  ...      交易所                  交易所英文名称
0  2026-09-04  000300  ...  深圳证券交易所  Shenzhen Stock Exchange
1  2026-09-04  000300  ...  深圳证券交易所  Shenzhen Stock Exchange
2  2026-09-04  000300  ...  深圳证券交易所  Shenzhen Stock Exchange
3  2026-09-04  000300  ...  深圳证券交易所  Shenzhen Stock Exchange
4  2026-09-04  000300  ...  深圳证券交易所  Shenzhen Stock Exchange

[5 rows x 9 columns]


In [5]:
print(cons.shape)
print(cons.columns)

(300, 9)
Index(['日期', '指数代码', '指数名称', '指数英文名称', '成分券代码', '成分券名称', '成分券英文名称', '交易所',
       '交易所英文名称'],
      dtype='str')


In [6]:
print(data_dir.resolve())
print(cons.shape)
print(cons.columns)

<PROJECT_ROOT>\data\raw
(300, 9)
Index(['日期', '指数代码', '指数名称', '指数英文名称', '成分券代码', '成分券名称', '成分券英文名称', '交易所',
       '交易所英文名称'],
      dtype='str')


In [7]:
if not cons_path.exists():
    cons.to_csv(cons_path, index=False, encoding="utf-8-sig")

In [8]:
codes = (
    cons["成分券代码"]
    .astype(str)
    .str.zfill(6)
    .tolist()
)

In [9]:
codes

Out[0]: 
['000001',
 '000002',
 '000063',
 '000100',
 '000157',
 '000166',
 '000301',
 '000333',
 '000338',
 '000408',
 '000425',
 '000538',
 '000568',
 '000596',
 '000617',
 '000625',
 '000630',
 '000651',
 '000657',
 '000708',
 '000725',
 '000768',
 '000776',
 '000792',
 '000807',
 '000858',
 '000895',
 '000938',
 '000963',
 '000975',
 '000977',
 '000988',
 '000999',
 '001280',
 '001391',
 '001965',
 '001979',
 '002001',
 '002027',
 '002028',
 '002049',
 '002050',
 '002074',
 '002142',
 '002179',
 '002202',
 '002230',
 '002236',
 '002241',
 '002304',
 '002311',
 '002352',
 '002353',
 '002371',
 '002384',
 '002415',
 '002422',
 '002460',
 '002463',
 '002466',
 '002475',
 '002493',
 '002532',
 '002558',
 '002594',
 '002600',
 '002602',
 '002625',
 '002648',
 '002709',
 '002714',
 '002736',
 '002837',
 '002916',
 '002920',
 '002938',
 '003816',
 '300014',
 '300015',
 '300033',
 '300059',
 '300122',
 '300124',
 '300251',
 '300274',
 '300308',
 '300316',
 '300394',
 '300408',
 '300413',
 

In [10]:
print(codes[:10])
print(len(codes))

['000001', '000002', '000063', '000100', '000157', '000166', '000301', '000333', '000338', '000408']
300


In [11]:
from src.data import get_stock_data

In [12]:
test_codes = codes[:3]

test_codes

Out[0]: ['000001', '000002', '000063']


In [13]:
test_data_list = []

for code in test_codes:
    try:
        df_temp = get_stock_data(code)

        if df_temp is not None:
            test_data_list.append(df_temp)
            print(code, "success", df_temp.shape)

        else:
            print(code, "empty")

    except Exception as e:
        print(code, "failed:", e)

000001 success (1455, 9)
000002 success (1455, 9)
000063 success (1454, 9)


In [14]:
test_data_list[0].head()

Out[0]: 
        date     open    close  ...  turnover        amount    code
0 2020-01-02  2607.73  2640.84  ...    0.0079  2.571196e+09  000001
1 2020-01-03  2651.38  2687.51  ...    0.0058  1.914496e+09  000001
2 2020-01-06  2661.92  2670.95  ...    0.0044  1.477930e+09  000001
3 2020-01-07  2679.98  2682.99  ...    0.0038  1.247047e+09  000001
4 2020-01-08  2660.41  2609.23  ...    0.0044  1.423609e+09  000001

[5 rows x 9 columns]


In [15]:
print(test_data_list[0]["date"].min())
print(test_data_list[0]["date"].max())

2020-01-02 00:00:00
2025-12-31 00:00:00


In [16]:
stock_data_list = []
failed_codes = []

for i, code in enumerate(codes, start=1):
    try:
        df_temp = get_stock_data(code)

        if df_temp is not None:
            stock_data_list.append(df_temp)
            print(
                f"[{i}/{len(codes)}]",
                code,
                "success",
                df_temp.shape
            )

        else:
            failed_codes.append(code)
            print(
                f"[{i}/{len(codes)}]",
                code,
                "empty"
            )

    except Exception as e:
        failed_codes.append(code)
        print(
            f"[{i}/{len(codes)}]",
            code,
            "failed:",
            e
        )

[1/300] 000001 success (1455, 9)
[2/300] 000002 success (1455, 9)
[3/300] 000063 success (1454, 9)
[4/300] 000100 success (1454, 9)
[5/300] 000157 success (1455, 9)
[6/300] 000166 success (1455, 9)
[7/300] 000301 success (1444, 9)
[8/300] 000333 success (1455, 9)
[9/300] 000338 success (1455, 9)
[10/300] 000408 success (1448, 9)
[11/300] 000425 success (1432, 9)
[12/300] 000538 success (1455, 9)
[13/300] 000568 success (1455, 9)
[14/300] 000596 success (1455, 9)
[15/300] 000617 success (1455, 9)
[16/300] 000625 success (1455, 9)
[17/300] 000630 success (1445, 9)
[18/300] 000651 success (1455, 9)
[19/300] 000657 success (1445, 9)
[20/300] 000708 success (1455, 9)
[21/300] 000725 success (1455, 9)
[22/300] 000768 success (1455, 9)
[23/300] 000776 success (1455, 9)
[24/300] 000792 success (1143, 9)
[25/300] 000807 success (1455, 9)
[26/300] 000858 success (1455, 9)
[27/300] 000895 success (1455, 9)
[28/300] 000938 success (1455, 9)
[29/300] 000963 success (1455, 9)
[30/300] 000975 success

In [17]:
print("成功数量:", len(stock_data_list))
print("失败数量:", len(failed_codes))
print("失败股票:", failed_codes)

成功数量: 300
失败数量: 0
失败股票: []


In [18]:
panel = pd.concat(
    stock_data_list,
    ignore_index=True
)

In [19]:
print(panel.shape)
print("股票数量:", panel["code"].nunique())
print("开始日期:", panel["date"].min())
print("结束日期:", panel["date"].max())

(416388, 9)
股票数量: 300
开始日期: 2020-01-02 00:00:00
结束日期: 2025-12-31 00:00:00


In [20]:
print("缺失值：")
print(panel.isna().sum())

print("重复的 code-date 行数：")
print(
    panel.duplicated(
        subset=["code", "date"]
    ).sum()
)

缺失值：
date        0
open        0
close       0
high        0
low         0
volume      0
turnover    0
amount      0
code        0
dtype: int64
重复的 code-date 行数：
0


In [21]:
price_path = data_dir / "csi300_prices_2020_2025.parquet"
if not price_path.exists():
    if failed_codes:
        raise RuntimeError(f"Incomplete download: {failed_codes}; refusing to save a partial panel")
    panel.to_parquet(price_path, index=False)
else:
    print("Existing frozen panel preserved")

Existing frozen panel preserved


In [22]:
panel_test = pd.read_parquet(
    data_dir / "csi300_prices_2020_2025.parquet"
)

print(panel_test.shape)
print(panel_test["code"].nunique())
panel_test.head()

(416388, 9)
300
Out[0]: 
        date     open    close  ...  turnover        amount    code
0 2020-01-02  2607.73  2640.84  ...    0.0079  2.571196e+09  000001
1 2020-01-03  2651.38  2687.51  ...    0.0058  1.914496e+09  000001
2 2020-01-06  2661.92  2670.95  ...    0.0044  1.477930e+09  000001
3 2020-01-07  2679.98  2682.99  ...    0.0038  1.247047e+09  000001
4 2020-01-08  2660.41  2609.23  ...    0.0044  1.423609e+09  000001

[5 rows x 9 columns]
